# QDMC Rare Event Sampling — Report Figures

This notebook recreates the three result figures from the project report:

- **Figure 5** — AI+RES surface-temperature trajectories (left) and the final-day
  temperature distribution (right), DNS vs AI+RES.
- **Figure 6a** — distribution of the observable (final 7-day regional-average
  surface temperature) for DNS and AI+RES, with Johnson-SU fits.
- **Figure 6b** — return-period curves (in years) using the corrected DMC
  importance-sampling estimator.

It reads the run outputs written by `run.py` into `outputs/`. Set `SCHEME` and
`RUN_DIR` below to point at the run you want to analyse.


In [ ]:
import json
import math
import pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import johnsonsu

# ── Which run to analyse ──────────────────────────────────────────────────
RUN_DIR = Path.cwd()              # the res/ directory (where config.json lives)
SCHEME = "ck"                     # AI+RES schedule used (run.py --scheme); "ck" is the report schedule
REGION_KEY = "nw_box"

DNS_DIR = RUN_DIR / "outputs" / "global_dns_output"
AI_DIR = RUN_DIR / "outputs" / f"ai_res_{REGION_KEY}_{SCHEME}_output"

# To instead render the original report data (old emulator, 400x100), uncomment:
#   DNS_DIR = Path("/home/nishidh/res_aiemulator_sem8/res/report_data/global_dns_large_output")
#   AI_DIR  = Path("/home/nishidh/res_aiemulator_sem8/res/report_data/ai_res_nw_box_ck_large_output")

with open(RUN_DIR / "config.json") as f:
    config = json.load(f)

REGION = config["TRACKED_REGIONS"][REGION_KEY]
L = config["L"]
SCHEDULE = config.get(f"C_schedule_{SCHEME}", [])
SCHEDULE_LABEL = "[" + ", ".join(f"{c:.1f}" for c in SCHEDULE) + "]"
REGION_TITLE = "Region bounds: 26.5N-32.1N, 70.3E-75.9E"

# Report colour scheme: DNS black, AI+RES blue.
C_DNS, C_AI = "#222222", "#2040ff"
C_DNS_LIGHT, C_AI_LIGHT = "#9fa3a7", "#9fb4ff"

print(f"DNS dir: {DNS_DIR}")
print(f"AI+RES dir: {AI_DIR}  (scheme {SCHEME} = {SCHEDULE_LABEL})")

In [ ]:
def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def region_series_from_walker(walker):
    """Daily regional-mean surface temperature (channel 0) over the target box."""
    out = []
    for day in walker.get("trajectory", []):
        if isinstance(day, np.ndarray) and day.ndim == 3 and day.shape[0] > 0:
            out.append(float(np.mean(
                day[0, REGION["lat_start"]:REGION["lat_end"], REGION["lon_start"]:REGION["lon_end"]]
            )))
    return np.asarray(out, dtype=float)


def final_value_from_walker(walker):
    """The observable A_{L,t_f}: final 7-day regional-average surface temperature."""
    if "A_L_tf" in walker:
        return float(walker["A_L_tf"])
    by_region = walker.get("A_L_tf_by_region") or {}
    if REGION_KEY in by_region:
        return float(by_region[REGION_KEY])
    series = region_series_from_walker(walker)
    return float(np.mean(series[-L:])) if len(series) else np.nan


def stack_series(series_list):
    if not series_list:
        return np.empty((0, 0), dtype=float)
    max_len = max(len(s) for s in series_list)
    out = np.full((len(series_list), max_len), np.nan, dtype=float)
    for i, s in enumerate(series_list):
        out[i, :len(s)] = s
    return out


def raw_importance_weights(walkers):
    """Unnormalised IS weights exp(-log_weight) that de-bias the resampled ensemble."""
    lw = np.asarray([float(w.get("log_weight", 0.0)) for w in walkers], dtype=float)
    finite = np.isfinite(lw)
    if not finite.all():
        lw = np.where(finite, lw, float(np.nanmax(lw[finite])) if finite.any() else 0.0)
    return np.exp(-lw)


# ── Load DNS + AI+RES runs ────────────────────────────────────────────────
dns_walkers = load_pickle(DNS_DIR / "dns_baseline_global.pkl")
ai_walkers = load_pickle(AI_DIR / "ai_res_final_walkers.pkl")
ai_global_weights = np.asarray(load_pickle(AI_DIR / "global_weights.pkl"), dtype=float)

dns_values = np.asarray([final_value_from_walker(w) for w in dns_walkers], dtype=float)
ai_values = np.asarray([final_value_from_walker(w) for w in ai_walkers], dtype=float)
dns_series = stack_series([region_series_from_walker(w) for w in dns_walkers])
ai_series = stack_series([region_series_from_walker(w) for w in ai_walkers])
ai_raw_weights = raw_importance_weights(ai_walkers)
N = len(ai_values)

print(f"DNS walkers: {len(dns_values)} | AI+RES walkers: {len(ai_values)}")
print(f"Mean observable — DNS: {np.mean(dns_values):.3f} K | "
      f"AI+RES: {np.mean(ai_values):.3f} K | shift: {np.mean(ai_values) - np.mean(dns_values):+.3f} K")

## Figure 5 — Trajectories and final-day distribution

Left: every AI+RES walker's daily regional-mean surface-temperature trajectory
(blue), with the ensemble mean overlaid. DNS trajectories are omitted for clarity
(the report does the same), but the DNS final-day distribution appears on the
right. Right: the final-day temperature distribution for DNS (black) and AI+RES
(blue), as step histograms with Johnson-SU fits.


In [ ]:
# Final-day temperature for each walker (last finite day of its trajectory).
dns_final = np.array([s[np.isfinite(s)][-1] for s in dns_series if np.any(np.isfinite(s))])
ai_final = np.array([s[np.isfinite(s)][-1] for s in ai_series if np.any(np.isfinite(s))])

all_final = np.concatenate([dns_final, ai_final])
bins = np.linspace(all_final.min() - 0.4, all_final.max() + 0.4, 18)
bin_w = float(bins[1] - bins[0])
domain = np.linspace(bins[0], bins[-1], 500)

fig, (ax1, ax2) = plt.subplots(
    1, 2, figsize=(12, 5.6), sharey=True,
    gridspec_kw={"width_ratios": [3, 1.2], "wspace": 0.05},
)
fig.suptitle(REGION_TITLE, fontsize=14)

# Left: AI+RES trajectories (blue) + mean.
for s in ai_series:
    m = np.isfinite(s)
    if m.any():
        ax1.plot(np.arange(len(s))[m], s[m], color=C_AI_LIGHT, alpha=0.08, linewidth=0.7)
ax1.plot(np.arange(ai_series.shape[1]), np.nanmean(ai_series, axis=0), color=C_AI, linewidth=2.4,
         label=f"AI+RES trajectories {SCHEDULE_LABEL}")
ax1.set_xlim(left=0)
ax1.set_title("Surface temperature trajectories")
ax1.set_xlabel("Simulation day")
ax1.set_ylabel("Surface temperature (K)")
ax1.legend(frameon=False, loc="upper left", fontsize=9)

# Right: final-day distribution (horizontal), DNS + AI+RES, step hist + Johnson-SU fit.
for vals, color, label in [(dns_final, C_DNS, "DNS"), (ai_final, C_AI, f"AI+RES {SCHEDULE_LABEL}")]:
    ax2.hist(vals, bins=bins, weights=np.full(len(vals), 1.0 / N), orientation="horizontal",
             histtype="step", linewidth=2.0, color=color, label=label)
    a, b, loc, scale = johnsonsu.fit(vals)
    ax2.plot(johnsonsu.pdf(domain, a, b, loc, scale) * bin_w, domain,
             color=color, linestyle="--", linewidth=1.8)
ax2.set_title("Final day distribution")
ax2.set_xlabel("Fraction of walkers")
ax2.tick_params(axis="y", which="both", left=False)
ax2.legend(frameon=False, loc="lower right", fontsize=9)

plt.tight_layout()
plt.show()

## Figure 6a — Observable distribution

Distribution of the observable (final 7-day regional-average surface temperature)
for DNS and AI+RES, with Johnson-SU fits. The rightward shift of the AI+RES
distribution is the rare-event enrichment achieved by resampling.


In [ ]:
all_vals = np.concatenate([dns_values, ai_values])
bins6 = np.linspace(all_vals.min() - 0.4, all_vals.max() + 0.4, 18)
bin_w6 = float(bins6[1] - bins6[0])
domain6 = np.linspace(bins6[0], bins6[-1], 500)

fig, ax = plt.subplots(figsize=(8.6, 5.6))
fig.suptitle("7-day regional average surface temperature distribution", fontsize=14)
for vals, color, label in [(dns_values, C_DNS, "DNS"), (ai_values, C_AI, f"AI+RES {SCHEDULE_LABEL}")]:
    ax.hist(vals, bins=bins6, weights=np.full(len(vals), 1.0 / N),
            histtype="step", linewidth=2.0, color=color, label=label)
    a, b, loc, scale = johnsonsu.fit(vals)
    ax.plot(domain6, johnsonsu.pdf(domain6, a, b, loc, scale) * bin_w6,
            color=color, linestyle="--", linewidth=1.8)
ax.set_xlabel("7-day regional average temperature (K)")
ax.set_ylabel("Fraction of walkers")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

print(f"Mean shift (AI+RES - DNS): {np.mean(ai_values) - np.mean(dns_values):+.2f} K")

## Figure 6b — Return-period curves

Return period (in years) versus the observable, for DNS and AI+RES. DNS uses the
empirical rank-based estimator; AI+RES uses the corrected Diffusion-Monte-Carlo
(DMC) estimator, where the normaliser is `prod(global_weights) / N` applied to the
cumulative raw importance weights. This lets a 400-walker AI+RES ensemble probe
return periods of order 10⁴ years — far beyond what 400 DNS walkers reach.


In [ ]:
MAX_RETURN_PERIOD = 1.2e4

# DNS: empirical exceedance from descending ranks.
dns_sorted = np.sort(dns_values)[::-1]
dns_probs = np.arange(1, len(dns_sorted) + 1) / len(dns_sorted)
dns_rp = 1.0 / dns_probs
dns_show = dns_rp <= MAX_RETURN_PERIOD

# AI+RES: corrected DMC tail probabilities.
dmc_constant = float(np.prod(ai_global_weights)) / N
order = np.argsort(ai_values)[::-1]
ai_sorted = ai_values[order]
ai_tail = dmc_constant * np.cumsum(ai_raw_weights[order])
mask = np.isfinite(ai_tail) & (ai_tail > 0)
ai_rp = (1.0 / ai_tail[mask])[::-1]
ai_vals_curve = ai_sorted[mask][::-1]
ai_show = ai_rp <= MAX_RETURN_PERIOD

fig, ax = plt.subplots(figsize=(6.0, 5.0))
fig.suptitle(REGION_TITLE.replace("Region bounds:", "Region:"), fontsize=13)
ax.scatter(dns_rp[dns_show], dns_sorted[dns_show], s=10, color=C_DNS, alpha=0.9,
           linewidths=0.0, label=f"DNS, N={len(dns_values)}")
ax.plot(ai_rp[ai_show], ai_vals_curve[ai_show], color=C_AI, linewidth=2.2,
        label=f"AI+RES, N={N}")
ax.set_xscale("log")
ax.set_xlim(1.0, MAX_RETURN_PERIOD)
ax.set_xlabel("Return period (years)")
ax.set_ylabel("Surface temperature (K)")
ax.legend(frameon=False, loc="lower right", fontsize=9)
ax.grid(True, which="major", alpha=0.18)
plt.tight_layout()
plt.show()

## Summary statistics

In [ ]:
def exceedance_dns(t):
    return float(np.mean(dns_values > t))

def exceedance_ai(t):
    return float(dmc_constant * np.sum(ai_raw_weights * (ai_values > t)))

def rp(p):
    return float("inf") if p <= 0 else 1.0 / p

print(f"{'Threshold (K)':>14} | {'DNS return (yr)':>16} | {'AI+RES return (yr)':>18}")
print("-" * 56)
for q in (0.90, 0.95, 0.99):
    t = float(np.quantile(dns_values, q))
    print(f"{t:>14.3f} | {rp(exceedance_dns(t)):>16.1f} | {rp(exceedance_ai(t)):>18.1f}")
for extra in (2.0, 3.0):
    t = float(np.mean(dns_values) + extra)
    print(f"{t:>14.3f} | {rp(exceedance_dns(t)):>16.1f} | {rp(exceedance_ai(t)):>18.1f}")

print(f"\nDNS mean:    {np.mean(dns_values):.3f} K  (std {np.std(dns_values, ddof=1):.3f})")
print(f"AI+RES mean: {np.mean(ai_values):.3f} K  (std {np.std(ai_values, ddof=1):.3f})")
print(f"Mean shift:  {np.mean(ai_values) - np.mean(dns_values):+.3f} K")
print(f"Cumulative enrichment (prod global_weights): {np.prod(ai_global_weights):.3e}")